In [1]:
import pandas as pd

wages = pd.DataFrame({
    "occ_code": ["15-1252", "29-1141", "41-2011", "19-3011", "53-3032", "25-2021", "11-1011"],
    "occupation": ["Software Developers", "Registered Nurses", "Cashiers", "Economists", "Truck Drivers", "Elementary Teachers", "Chief Executives"],
    "median_wage": [132270, 86070, 29720, 115730, 54320, 66470, 206420]
})

usage = pd.DataFrame({
    "soc": ["15-1252.00", "29-1141.00", "19-3011.00 ", "25-2021.00", "13-2011.00", "15-2051.00"],
    "usage_share": [0.182, 0.031, 0.094, 0.055, 0.048, 0.121]
})

In [17]:
wages

,occ_code,occupation,median_wage
0,15-1252,Software Developers,132270
1,29-1141,Registered Nurses,86070
2,41-2011,Cashiers,29720
3,19-3011,Economists,115730
4,53-3032,Truck Drivers,54320
5,25-2021,Elementary Teachers,66470
6,11-1011,Chief Executives,206420


In [18]:
usage

,soc,usage_share,occ_code
0,15-1252.00,0.182,15-1252
1,29-1141.00,0.031,29-1141
2,19-3011.00,0.094,19-3011
3,25-2021.00,0.055,25-2021
4,13-2011.00,0.048,13-2011
5,15-2051.00,0.121,15-2051


In [6]:
merged = pd.merge(wages, usage, left_on="occ_code", right_on="soc", how="inner")
merged

,occ_code,occupation,median_wage,soc,usage_share


In [7]:
print("wages:", len(wages))
print("usage:", len(usage))
print("merged:", len(merged))

wages: 7
usage: 6
merged: 0


In [8]:
set(wages["occ_code"]) & set(usage["soc"])

set()

In [9]:
usage["soc"].values

<StringArray>
[ '15-1252.00',  '29-1141.00', '19-3011.00 ',  '25-2021.00',  '13-2011.00',
  '15-2051.00']
Length: 6, dtype: str

In [10]:
usage["soc"].str.len()

0    10
1    10
2    11
3    10
4    10
5    10
Name: soc, dtype: int64

Differences in wages and usage keys:
- ".00" suffix
- Trailing spaces

In [11]:
usage["occ_code"] = usage["soc"].str.strip().str.replace(".00", "", regex=False)

In [13]:
set(wages["occ_code"]) & set(usage["occ_code"])

{'15-1252', '19-3011', '25-2021', '29-1141'}

In [14]:
wages["occ_code"].is_unique
usage["occ_code"].is_unique

True

In [15]:
merged = pd.merge(wages, usage, on="occ_code", how="inner")
merged

,occ_code,occupation,median_wage,soc,usage_share
0,15-1252,Software Developers,132270,15-1252.00,0.182
1,29-1141,Registered Nurses,86070,29-1141.00,0.031
2,19-3011,Economists,115730,19-3011.00,0.094
3,25-2021,Elementary Teachers,66470,25-2021.00,0.055


In [16]:
print("wages:", len(wages), "usage:", len(usage), "merged:", len(merged))

wages: 7 usage: 6 merged: 4


Notes:
- `usage["soc"]` contained ".00" and trailing spaces and `wages["occ_code"]` did not; stripped trailing spaces and removed ".00"; verified by checking overlap of `wages["occ_code"]` with cured `usage["occ_code"]` keys
- Match rate went from 0 of 7 to 4 of 7 after normalization
- The 3 unmatched wage rows (Cashiers, Truck Drivers, Chief Executives) and 2 unmatched usage rows (13-2011 Accountants, 15-2051 Data Scientists) are legitimate non-matches; Consequence: aggregates describe a compressed middle rather than full distribution because missing matches are two lowest-paid and highest-paid.

Question: What must you do to the usage side before a real merge with a table with thousands of rows, and which check proves you did it right?
Answer: The usage side has many rows per occupation, so before merging I must aggregate it to one row per occ_code with groupby. `.is_unique` is the check to confirm there are no longer duplicates.